# Train Model B - GPA Regressor

The original Model B (`modelB.pkl`) used `GradientBoostingRegressor` from sklearn, **not supported by m2cgen 0.10** for JavaScript transpilation.

This notebook retrains Model B using `XGBRegressor` (supported by m2cgen) on the same dataset and split, so performance can be compared fairly.

Output: `models/modelB.pkl` (~565 KB)

## 1. Imports & Config

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib
import warnings
from scipy import stats

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATA_PATH = os.path.join(REPO_ROOT, "datasets", "student_lifestyle_dataset.csv")
OUT_PATH  = os.path.join(REPO_ROOT, "models", "modelB.pkl")

# Column order MUST match with src.config.MODEL_B_BASE_FEATURES
FEATURE_ORDER = [
    "study_hours", "eca_hours", "sleep_hours",
    "social_hours", "physical_hours", "stress_level",
]

## 2. Load & Preprocess Data

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=["Student_ID"])
df = df.rename(columns={
    "Study_Hours_Per_Day":              "study_hours",
    "Extracurricular_Hours_Per_Day":    "eca_hours",
    "Sleep_Hours_Per_Day":              "sleep_hours",
    "Social_Hours_Per_Day":             "social_hours",
    "Physical_Activity_Hours_Per_Day":  "physical_hours",
    "Stress_Level":                     "stress_level",
    "GPA":                              "gpa",
})
df["stress_level"] = df["stress_level"].map({"Low": 0, "Moderate": 1, "High": 2}) # Manual mapping for further processing, since categorical column can not be processed

# Z-score cleaning as Z-score cleaning is less aggresive than IQR cleaning for small dataset
df = df[(np.abs(stats.zscore(df.select_dtypes("number"))) < 3).all(axis=1)]

print(f"Dataset after cleaning: {len(df)} rows")
df.head()

Dataset after cleaning: 1996 rows


,study_hours,eca_hours,sleep_hours,social_hours,physical_hours,gpa,stress_level
0,6.9,3.8,8.7,2.8,1.8,2.99,1
1,5.3,3.5,8.0,4.2,3.0,2.75,0
2,5.1,3.9,9.2,1.2,4.6,2.67,0
3,6.5,2.1,7.2,1.7,6.5,2.88,1
4,8.1,0.6,6.5,2.2,6.6,3.51,2


## 3. Train/Test Split

In [3]:
X = df[FEATURE_ORDER]
y = df["gpa"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Length of training set: {len(X_train)} & length of test set: {len(X_test)}")

Length of training set: 1397 & length of test set: 599


## 4. Hyperparameter Tuning (RandomizedSearchCV)

Search space is intentionally focused around XGBRegressor best-practice ranges:
- `n_estimators` is guarded to stay within m2cgen's transpilation limit
- `max_depth` kept shallow (2-5) to prevent overfitting on this ~2K-row dataset
- 20 iterations × 3-fold CV = 60 fits, typically under 3 minutes

Then, we obtain the hyperparameters (or params) to be used by the trained model.

In [4]:
from sklearn.model_selection import RandomizedSearchCV, KFold
import time

param_dist = {
    "n_estimators":      [200, 300, 400, 500, 600],
    "max_depth":         [2, 3, 4, 5],
    "learning_rate":     [0.01, 0.02, 0.035, 0.05, 0.07, 0.10],
    "subsample":         [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":  [0.6, 0.7, 0.8, 1.0],
    "min_child_weight":  [1, 3, 5],
    "reg_lambda":        [0.5, 1.0, 1.5, 2.0],
}

base_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

kf = KFold(n_splits=3, shuffle=True, random_state=42)

rscv = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=20,
    scoring="r2",
    cv=kf,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

rscv.fit(X_train, y_train)
print(f"Best CV R²  : {rscv.best_score_:.4f}")
print(f"Best params : {rscv.best_params_}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best CV R²  : 0.5017
Best params : {'subsample': 0.7, 'reg_lambda': 2.0, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 3, 'learning_rate': 0.01, 'colsample_bytree': 0.8}


## 5. Training

In [5]:
best = rscv.best_params_

model = xgb.XGBRegressor(
    n_estimators     = best["n_estimators"],
    max_depth        = best["max_depth"],
    learning_rate    = best["learning_rate"],
    subsample        = best["subsample"],
    colsample_bytree = best["colsample_bytree"],
    min_child_weight = best["min_child_weight"],
    reg_lambda       = best["reg_lambda"],
    objective        = "reg:squarederror",
    random_state     = 42,
    n_jobs           = -1,
    verbosity        = 0,
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

## 6. Evaluation

In [6]:
print("Model B - Evaluation Metrics:")
print(f"R2   : {r2_score(y_test, pred):.4f}")
print(f"MAE  : {mean_absolute_error(y_test, pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred)):.4f}")

Model B - Evaluation Metrics:
R2   : 0.5357
MAE  : 0.1637
RMSE : 0.2024


## 6b. Cross-Validation

5-fold KFold R² - confirms the compact regressor generalizes across splits.

In [7]:
from sklearn.model_selection import KFold, cross_val_score

def _build_model():
    return xgb.XGBRegressor(
        n_estimators     = best["n_estimators"],
        max_depth        = best["max_depth"],
        learning_rate    = best["learning_rate"],
        subsample        = best["subsample"],
        colsample_bytree = best["colsample_bytree"],
        min_child_weight = best["min_child_weight"],
        reg_lambda       = best["reg_lambda"],
        objective        = "reg:squarederror",
        random_state     = 42,
        n_jobs           = -1,
        verbosity        = 0,
    )

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(_build_model(), X, y, cv=kf, scoring="r2", n_jobs=-1)
print("Model B - Cross-Validation (5-fold):")
print(f"CV R2 : {scores.mean():.4f} +/- {scores.std():.4f}")


Model B - Cross-Validation (5-fold):
CV R2 : 0.5279 +/- 0.0249


## 7. Save Model

In [8]:
joblib.dump(model, OUT_PATH)
print(f"Saved model B successfully!")

Saved model B successfully!
